# PANTANAL-1 M02 — Kaggle Notebook Smoke

This is an **M02 smoke notebook**. It uses **synthetic labels only** (`synthetic_class_000` … `synthetic_class_233`).

- It does not use Kaggle competition data.
- It does not prove leaderboard readiness or model inference.
- It does not prove the notebook ran on Kaggle, active submission eligibility, real `sample_submission.csv` compatibility, CPU 90-minute runtime compliance, or a competition score.

For a real Kaggle submission notebook, the final output path must be `/kaggle/working/submission.csv`. M02 intentionally does not enable that path in the inline fallback because M02 is synthetic smoke only. A future milestone must add real Kaggle sample/schema alignment before enabling real submission output.

M02 intentionally writes to `tmp/submissions/m02_smoke_submission.csv` (not `/kaggle/working/submission.csv`). Local repo smoke must not create a root-level `submission.csv` in the repository.

If `pantanal_1` is not installed (typical when this notebook is copied into a blank Kaggle notebook), the notebook falls back to an inline synthetic implementation with verbose diagnostics.

In [ ]:
import os
import platform
import sys
from pathlib import Path

print("=== PANTANAL-1 M02 Kaggle Smoke Debug ===")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("cwd:", Path.cwd())
print("sys.path first entries:", sys.path[:10])
print("KAGGLE_KERNEL_RUN_TYPE:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
print("KAGGLE_URL_BASE:", os.environ.get("KAGGLE_URL_BASE"))

for path in [Path("/kaggle"), Path("/kaggle/input"), Path("/kaggle/working"), Path("tmp")]:
    print(f"{path}: exists={path.exists()}")
    if path.exists():
        try:
            print(f"{path} children:", [p.name for p in list(path.iterdir())[:20]])
        except Exception as exc:
            print(f"{path} listing failed:", repr(exc))

In [ ]:
from pathlib import Path

try:
    from pantanal_1.submission_contract import (
        build_zero_submission_rows,
        validate_submission_rows,
        write_submission_csv,
    )
    from pantanal_1.synthetic_schema import SYNTHETIC_CLASS_LABELS, SYNTHETIC_SOUNDSCAPE_STEMS

    SOURCE = "pantanal_1 package import"
except ModuleNotFoundError as exc:
    print("Package import failed:", repr(exc))
    print("Falling back to inline M02 synthetic smoke implementation.")

    import csv

    SOURCE = "inline fallback"

    SYNTHETIC_CLASS_LABELS = tuple(f"synthetic_class_{idx:03d}" for idx in range(234))
    SYNTHETIC_SOUNDSCAPE_STEMS = (
        "BC2026_Test_0001_S05_20250227_010002",
        "BC2026_Test_0002_S05_20250227_010003",
    )

    def build_segment_end_times(duration_seconds=60, window_seconds=5):
        if duration_seconds <= 0 or window_seconds <= 0:
            raise ValueError("duration_seconds and window_seconds must be positive")
        if duration_seconds % window_seconds != 0:
            raise ValueError("duration_seconds must be divisible by window_seconds")
        return list(range(window_seconds, duration_seconds + 1, window_seconds))

    def build_row_id(soundscape_stem, end_time_seconds):
        return f"{soundscape_stem}_{end_time_seconds}"

    def build_zero_submission_rows(soundscape_stems, class_labels):
        rows = []
        for stem in soundscape_stems:
            for end_time in build_segment_end_times():
                row = {"row_id": build_row_id(stem, end_time)}
                row.update({label: 0.0 for label in class_labels})
                rows.append(row)
        return rows

    def validate_submission_rows(rows, class_labels):
        if len(class_labels) != 234:
            raise ValueError(f"expected 234 class labels, got {len(class_labels)}")
        seen = set()
        expected_keys = {"row_id", *class_labels}
        for idx, row in enumerate(rows):
            keys = set(row)
            if keys != expected_keys:
                raise ValueError(f"row {idx} keys mismatch")
            row_id = row["row_id"]
            if row_id in seen:
                raise ValueError(f"duplicate row_id: {row_id}")
            seen.add(row_id)
            for label in class_labels:
                value = row[label]
                if not isinstance(value, (int, float)):
                    raise ValueError(f"{label} is not numeric")
                if not 0.0 <= float(value) <= 1.0:
                    raise ValueError(f"{label} out of range: {value}")

    def write_submission_csv(rows, output_path, class_labels):
        output_path = Path(output_path)
        if output_path.name != "m02_smoke_submission.csv":
            raise ValueError("M02 local smoke must write m02_smoke_submission.csv")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fieldnames = ["row_id", *class_labels]
        with output_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        return output_path


print("Import/bootstrap complete. SOURCE =", SOURCE)

In [ ]:
print("Implementation source:", SOURCE)
class_labels = list(SYNTHETIC_CLASS_LABELS)
stems = list(SYNTHETIC_SOUNDSCAPE_STEMS)
print("class label count:", len(class_labels))
print("first 5 labels:", class_labels[:5])
print("last 5 labels:", class_labels[-5:])
print("soundscape stems:", stems)

In [ ]:
rows = build_zero_submission_rows(stems, class_labels)
validate_submission_rows(rows, class_labels)
print("row count:", len(rows))
print("expected rows:", len(stems) * 12)
print("first row_id:", rows[0]["row_id"])
print("last row_id:", rows[-1]["row_id"])
print("first row column count:", len(rows[0]))
print("first row preview:", {k: rows[0][k] for k in list(rows[0])[:8]})

In [ ]:
output_path = Path("tmp/submissions/m02_smoke_submission.csv")
write_submission_csv(rows, output_path, class_labels)
print("wrote:", output_path)
print("exists:", output_path.exists())
print("size bytes:", output_path.stat().st_size if output_path.exists() else None)

with output_path.open("r", encoding="utf-8") as handle:
    for idx in range(3):
        line = handle.readline().rstrip()
        print(f"csv line {idx}:", line[:500])